<a href="https://colab.research.google.com/github/busycaesar/Attention_Attention_Everywhere/blob/Master/QKV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U -q keras-hub keras

In [2]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['KERAS_BACKEND'] = 'jax'

In [3]:
import keras, keras_hub

tokenizer = keras_hub.models.Gemma3Tokenizer.from_preset('gemma3_1b')
model = keras_hub.models.Gemma3Backbone.from_preset('gemma3_1b', dtype='float32')

100%|██████████| 1.86G/1.86G [00:39<00:00, 50.8MB/s]


In [4]:
sentence = 'The dog did not cross the road because it was'

In [14]:
import numpy as np

ids = [tokenizer.start_token_id] + [int(t) for t in keras.ops.convert_to_numpy(tokenizer(sentence))]
inputs = {'token_ids': np.array([ids]), 'padding_mask': np.ones((1, len(ids)), dtype='int32')}

tokens = [tokenizer.id_to_token(tid).replace('\u2581', ' ') for tid in inputs['token_ids'][0]]

it_idx = tokens.index(' it')
dog_idx = tokens.index(' dog')
road_idx = tokens.index(' road')

## Tokens become vectors​

In [9]:
from types import SimpleNamespace

outputs = SimpleNamespace(last_hidden_state=keras.ops.convert_to_numpy(model(inputs)))

for idx, tok in enumerate(tokens[1:], start=1):
    word = tok.strip()
    vec = outputs.last_hidden_state[0, idx][:5]
    values = ', '.join(f'{v:+.3f}' for v in vec.tolist())
    print(f'{word} = [{values}, ...]\n')

The = [+1.918, -1.409, -1.673, -0.984, -1.070, ...]

dog = [-0.460, -2.084, +1.492, -1.025, -0.566, ...]

did = [-0.794, -1.239, -0.174, +0.384, +0.031, ...]

not = [-2.786, -1.724, +1.938, -0.470, -0.153, ...]

cross = [+0.781, +2.134, +1.993, -0.296, +0.206, ...]

the = [-0.300, -0.525, +0.829, +1.689, +0.812, ...]

road = [+1.535, -0.742, +1.909, -0.740, -0.499, ...]

because = [+1.279, -0.727, +0.893, -0.628, -0.075, ...]

it = [+0.983, +2.105, +0.507, -1.104, -1.146, ...]

was = [+0.030, -1.153, +1.495, -1.200, -0.966, ...]



## From output vector to next token prediction​

In [11]:
import numpy as np

h = keras.ops.convert_to_numpy(model(inputs))[0, -1]

print(f'Output vector: [{", ".join(f"{x:+.3f}" for x in h[:8].tolist())}, ...]   ({len(h)} numbers)\n')

top_n_tokens = 100

logits = h @ keras.ops.convert_to_numpy(model.token_embedding.embeddings).T
vocab_size = model.vocabulary_size
probs  = np.exp(logits - logits.max()); probs = probs / probs.sum()
top    = np.argsort(probs)[::-1][:top_n_tokens]

print(f'Scored against all {vocab_size} tokens. top {top_n_tokens}:\n')
for i in top:
    print(f'  {tokenizer.detokenize([int(i)]):<14} {probs[i]*100:5.5f}%')

Output vector: [+0.030, -1.153, +1.495, -1.200, -0.966, +3.993, +1.647, -3.420, ...]   (1152 numbers)

Scored against all 262144 tokens. top 100:

   afraid        16.80490%
   a             8.85821%
   not           5.79811%
   scared        5.27416%
   on            3.71486%
   too           3.61922%
   in            3.47888%
   frightened    3.16535%
   blind         2.37096%
   the           2.00281%
   black         1.20381%
   dark          0.94300%
   an            0.94118%
   tired         0.77216%
   very          0.76234%
   crossing      0.75869%
   busy          0.69424%
   running       0.69097%
   so            0.68110%
   going         0.63443%
   hungry        0.63129%
   being         0.61386%
   cold          0.61135%
   unaware       0.57826%
   under         0.55210%
   safe          0.54856%
   distracted    0.53675%
   aware         0.51598%
   following     0.48771%
   trying        0.48464%
   just          0.45203%
   looking       0.44677%
   worried       0.4

## QK: does the ask match the offer?​

In [12]:
import numpy as np
from keras import ops
from types import SimpleNamespace

qkv_store = {}
attentions = []
hidden_states = []

blocks = model.transformer_layers
n_q, n_kv = model.num_query_heads, model.num_key_value_heads

# Keras has no forward hooks, so we run the layers one by one
# and read Q, K, V out of each attention layer on the way.
x = model.token_embedding(inputs['token_ids']) * ops.sqrt(ops.cast(model.hidden_dim, 'float32'))

for L, block in enumerate(blocks):
    hidden_states.append(ops.convert_to_numpy(x))

    attn = block.attention
    xn = block.pre_attention_norm(x)
    Q = attn._apply_rope(attn.query_norm(attn.query_dense(xn)), 0)
    K = attn._apply_rope(attn.key_norm(attn.key_dense(xn)), 0)
    V = attn.value_dense(xn)
    qkv_store[L] = tuple(ops.convert_to_numpy(t)[0] for t in (Q, K, V))   # (tokens, heads, head_dim)

    Qn, Kn, _ = qkv_store[L]
    T = Qn.shape[0]
    scores = np.einsum('thd,shd->hts', Qn, np.repeat(Kn, n_q // n_kv, axis=1)) / np.sqrt(model.head_dim)
    scores = np.where(np.tril(np.ones((T, T), bool)), scores, -np.inf)
    w = np.exp(scores - scores.max(-1, keepdims=True))
    attentions.append((w / w.sum(-1, keepdims=True))[None])             # (1, heads, tokens, tokens)

    x = block(x, padding_mask=inputs['padding_mask'])

hidden_states.append(ops.convert_to_numpy(x))
outputs = SimpleNamespace(attentions=attentions)

In [15]:
import numpy as np

L, H   = 6, 0
d_head = model.head_dim
Q, K, _ = qkv_store[L]
kv     = H // (n_q // n_kv)

q_it   = Q[it_idx,   H]
k_dog  = K[dog_idx,  kv]
k_road = K[road_idx, kv]

N = 5

def row(name, v):
    print(f'  {name:<9} = [{", ".join(f"{x:+.3f}" for x in v[:N].tolist())}, ... ]   ({len(v)} numbers)')

print(f'Gemma 3\n')
row('q_it',   q_it)
row('k_dog',  k_dog)
row('k_road', k_road)

for nm, k in [('dog', k_dog), ('road', k_road)]:
    print(f'\n  q_it · k_{nm}')
    print('    ' + ' + '.join(f'({q_it[i]:+.3f})({k[i]:+.3f})' for i in range(N)) + ' + ...')
    dot = np.dot(q_it, k).item()
    print(f'    = {dot:+.4f}      (all {len(q_it)} terms)')

Gemma 3

  q_it      = [-1.274, +0.702, +1.210, -0.462, +0.879, ... ]   (256 numbers)
  k_dog     = [-0.855, +0.653, +1.007, -0.729, -0.001, ... ]   (256 numbers)
  k_road    = [-0.020, +0.101, -0.126, +0.840, +0.394, ... ]   (256 numbers)

  q_it · k_dog
    (-1.274)(-0.855) + (+0.702)(+0.653) + (+1.210)(+1.007) + (-0.462)(-0.729) + (+0.879)(-0.001) + ...
    = +138.6383      (all 256 terms)

  q_it · k_road
    (-1.274)(-0.020) + (+0.702)(+0.101) + (+1.210)(-0.126) + (-0.462)(+0.840) + (+0.879)(+0.394) + ...
    = +105.4654      (all 256 terms)


## The full trace​

In [16]:
print('Gemma 3 loaded.')
print(f'Layers : {model.num_layers}')
print(f'Heads  : {model.num_query_heads}  (query heads, sharing {model.num_key_value_heads} key/value head(s))')
print(f'Embedding dimension: {model.hidden_dim}')
print(f'Q, K, V per head: {model.head_dim}')

Gemma 3 loaded.
Layers : 26
Heads  : 4  (query heads, sharing 1 key/value head(s))
Embedding dimension: 1152
Q, K, V per head: 256


In [21]:
def fmt(v, n=4):
    return '[' + ', '.join(f'{x:+.3f}' for x in v[:n].tolist()) + ', ...]'

def fmt_ends(v, n=3):
    h = ', '.join(f'{x:+.3f}' for x in v[:n].tolist())
    t = ', '.join(f'{x:+.3f}' for x in v[-n:].tolist())
    return f'[{h}, ... , {t}]   (len {len(v)})'

hs = hidden_states

n_layers = model.num_layers
n_heads  = model.num_query_heads
d_head   = model.head_dim
prior    = list(range(1, it_idx + 1))
names    = [tokens[i].strip() for i in prior]

print('='*100)
print('Starting vector for "it"  (vector embedding x sqrt(d_model); position is added inside attention by RoPE)')
print('='*100)
print(f'  x_it = {fmt(hs[0][0, it_idx], 10)}   ({model.hidden_dim} numbers total)')
print('\n  note: each layer also has an MLP sublayer that adds to x_it.')
print('        the tables below trace the attention half only.\n')

for L in range(n_layers):
    print('\n' + '='*100)
    print(f'LAYER {L}')
    print('='*100)

    hdr = f'{"Head":<6}' + ''.join(f'{n:>10}' for n in names)
    print(hdr); print('-'*len(hdr))
    for H in range(n_heads):
        w = outputs.attentions[L][0][H][it_idx][prior]
        print(f'{H:<6}' + ''.join(f'{v:>10.3f}' for v in w.tolist()))

    _, _, V = qkv_store[L]
    head_outs = []
    for H in range(n_heads):
        w  = outputs.attentions[L][0][H][it_idx]
        Vh = V[:, H // (n_q // n_kv)]
        head_outs.append((w[:, None] * Vh).sum(0))

    # ---- show the formula with real numbers for head 0 ----
    w0  = outputs.attentions[L][0][0][it_idx]
    V0  = V[:, 0]
    print(f'\n  head output = sum of (attention weight x value vector)')
    print(f'  head 0, written out:\n')
    for i in prior:
        print(f'      {w0[i]:+.3f}  x  v_{names[i - 1]:<8} {fmt(V0[i], 3)}')
    print(f'      {"-"*54}')
    print(f'      sum     = {fmt(head_outs[0], 3)}   ({d_head} numbers)')

    z = np.concatenate(head_outs)
    print(f'\n  all {n_heads} heads flattened:')
    print(f'      z = {fmt_ends(z)}')

    Wo = keras.ops.convert_to_numpy(blocks[L].attention.output_dense.kernel).reshape(n_heads * d_head, -1)

    print(f'\n  W^O  — one per layer, shared by all {n_heads} heads   (top-left 4x4 of {Wo.shape[0]}x{Wo.shape[1]})')
    for r in range(4):
        print('      ' + ''.join(f'{Wo[r, c]:>9.3f}' for c in range(4)))

    attn_out = z @ Wo
    print(f'\n  z @ W^O  — each output slot is z dotted with one column of W^O')
    terms = ' + '.join(f'({z[k]:+.3f})({Wo[k,0]:+.3f})' for k in range(3))
    print(f'      slot 0 = {terms} + ...')
    print(f'             = {attn_out[0]:+.3f}')
    print(f'      attention output = {fmt(attn_out)}')

    attn_out = keras.ops.convert_to_numpy(blocks[L].post_attention_norm(attn_out[None, None]))[0, 0]
    print(f'      after post-attention norm = {fmt(attn_out)}')

    x_old = hs[L][0, it_idx]
    print(f'\n  new x_it = old x_it + attention output')
    print(f'    old x_it     {fmt(x_old)}')
    print(f'    + attn output  {fmt(attn_out)}')
    print(f'    ' + '-'*46)
    print(f'    = new x_it     {fmt(x_old + attn_out)}')

Starting vector for "it"  (vector embedding x sqrt(d_model); position is added inside attention by RoPE)
  x_it = [-0.965, -0.580, -0.143, +0.149, -0.866, -1.036, +1.077, -0.402, +0.982, +0.630, ...]   (1152 numbers total)

  note: each layer also has an MLP sublayer that adds to x_it.
        the tables below trace the attention half only.


LAYER 0
Head         The       dog       did       not     cross       the      road   because        it
------------------------------------------------------------------------------------------------
0          0.003     0.009     0.014     0.005     0.004     0.021     0.028     0.695     0.091
1          0.127     0.009     0.030     0.033     0.011     0.078     0.018     0.062     0.070
2          0.162     0.073     0.062     0.038     0.042     0.054     0.160     0.086     0.013
3          0.068     0.519     0.045     0.037     0.034     0.023     0.023     0.027     0.046

  head output = sum of (attention weight x value vector)
  head 